In [16]:
import numpy as np
import gvar as gv
import pickle

In [ ]:
def compute_continuum_energy(
    M_fit_jkn:  np.ndarray,
    nmax:    int,
    ns:     int,
) -> np.ndarray:
    """
    E(p) = sqrt(p^2 + M^2)
    where p^2 = (2*pi/Ns)^2 * n^2
    """
    p_squared = (2.0 * np.pi / ns) ** 2 * nmax
    return np.sqrt(p_squared + M_fit_jkn ** 2)

In [20]:
# Lattice spatial extent (D251)
ensemble = "D251"
ns = 64  

with open(f"/hdd/data/fit_results/{ensemble}/nucleon_energies/{ensemble}_E0_jkn_mom00_tzero03.pkl", "rb") as f:
    M_jkn = pickle.load(f)
n_res = len(M_jkn)

print(f"{'n^2':>4} {'E_fit':>15} {'E_cont':>20}")
print("-" * 64)

for nmax in [0, 1, 2, 3, 4, 5, 6, 8]:
    with open(f"/hdd/data/fit_results/{ensemble}/nucleon_energies/{ensemble}_E0_jkn_mom0{nmax}_tzero03.pkl", "rb") as f:
        E0_fit_jkn = pickle.load(f)

    # Continuum prediction per resample, using the SAME resample index for M
    E0_cont_jkn = compute_continuum_energy(M_jkn=M_jkn, nmax=nmax, ns=ns)

    # Central values
    E0_fit_cen  = float(np.mean(E0_fit_jkn))
    E0_cont_cen = float(np.mean(E0_cont_jkn))

    diff_jkn    = E0_fit_jkn - E0_cont_jkn
    diff_cen    = float(np.mean(diff_jkn))
    diff_err    = float(np.sqrt(n_res - 1) * np.std(diff_jkn, ddof=0))

    # Individual errors
    E0_fit_err  = float(np.sqrt(n_res - 1) * np.std(E0_fit_jkn,  ddof=0))
    E0_cont_err = float(np.sqrt(n_res - 1) * np.std(E0_cont_jkn, ddof=0))

    n_sigma = diff_cen / diff_err if diff_err > 0 else float("inf")


    print(f"{nmax:>4d}  "
          f"{str(gv.gvar(E0_fit_cen,  E0_fit_err)):>18}  "
          f"{str(gv.gvar(E0_cont_cen, E0_cont_err)):>18}  "
          f"{n_sigma:>10.2f}")

 n^2           E_fit               E_cont
----------------------------------------------------------------
   0         0.33821(58)         0.33821(58)         inf
   1         0.35197(61)         0.35217(56)       -0.81
   2         0.36517(71)         0.36560(54)       -0.97
   3         0.37775(89)         0.37855(52)       -1.17
   4          0.3898(12)         0.39107(50)       -1.18
   5          0.4010(14)         0.40321(49)       -1.71
   6          0.4112(18)         0.41499(47)       -2.22
   8          0.4319(26)         0.43760(45)       -2.18
